# QLoRA Fine-Tuning: Qwen2.5-1.5B-Instruct → Hint-Only Coding Assistant

This notebook fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with **QLoRA** (4-bit NF4 quantization + LoRA adapters) so the model gives **hints instead of full solution code** for a coding-assessment context.

Pipeline:
1. Install deps
2. Define **all hyperparameters** up front
3. Quantize base model with `bitsandbytes` (4-bit)
4. Load & format `dataset.jsonl` (chat-format, 300 examples)
5. Attach LoRA adapters, train with `SFTTrainer`
6. Evaluate: base model vs fine-tuned model, side-by-side, plus loss/leakage metrics

> Upload your `dataset.jsonl` (chat-message format as in your prompt) into the Colab file browser before running the dataset cell, or mount Drive.

## 0. Install dependencies
Run once. Restart runtime if prompted after `bitsandbytes`/`transformers` upgrade.

In [ ]:
!pip install -q -U "transformers>=4.44.0" "accelerate>=0.33.0" "peft>=0.12.0" \
    "bitsandbytes>=0.43.1" "trl>=0.9.6" "datasets>=2.20.0" "evaluate" "sentencepiece" "scipy"

In [ ]:
import torch, transformers, peft, trl, bitsandbytes as bnb
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('trl:', trl.__version__)
print('bitsandbytes:', bnb.__version__)
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > GPU'

## 1. Hyperparameters (single source of truth)
Everything downstream reads from this config — change values here only.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # --- Model / data ---
    base_model_id: str = "Qwen/Qwen2.5-1.5B-Instruct"
    dataset_path: str = "dataset.jsonl"        # path to your uploaded file
    output_dir: str = "qwen2.5-1.5b-hint-qlora"
    max_seq_length: int = 1024
    val_split_ratio: float = 0.10               # held-out eval set from the 300 examples
    seed: int = 42

    # --- Quantization (BitsAndBytes) ---
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_compute_dtype: str = "bfloat16"     # bf16 on Ampere+ (T4 falls back to fp16 automatically below)
    bnb_4bit_use_double_quant: bool = True

    # --- LoRA ---
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])
    lora_bias: str = "none"

    # --- Training ---
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8         # effective batch size = 16
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 0.3
    optim: str = "paged_adamw_8bit"
    logging_steps: int = 5
    eval_strategy: str = "epoch"
    save_strategy: str = "epoch"
    save_total_limit: int = 2
    packing: bool = False                        # keep False so per-example structure/labels stay clean
    gradient_checkpointing: bool = True

    # --- Generation (for evaluation) ---
    max_new_tokens: int = 200
    gen_temperature: float = 0.7
    gen_top_p: float = 0.9

cfg = Config()
cfg

In [ ]:
import random, numpy as np, torch
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)

## 2. Quantize base model with BitsAndBytes (4-bit NF4)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = getattr(torch, cfg.bnb_4bit_compute_dtype)
# T4 GPUs (free Colab tier) don't support bf16 compute well -> fall back to fp16 automatically
if not torch.cuda.is_bf16_supported():
    compute_dtype = torch.float16
    print("bf16 not supported on this GPU -> using fp16 compute dtype")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=cfg.bnb_4bit_use_double_quant,
)

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print(model)

## 3. Load and format the dataset
Expects `dataset.jsonl` where each line is `{"messages": [{"role": ..., "content": ...}, ...]}` exactly as in your prompt. We apply the tokenizer's chat template, split train/val, and sanity-check a few rows.

In [ ]:
import json, os
from datasets import load_dataset, Dataset

assert os.path.exists(cfg.dataset_path), f"Upload {cfg.dataset_path} to the Colab working directory first."

raw = load_dataset("json", data_files=cfg.dataset_path, split="train")
print(f"Loaded {len(raw)} examples")

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

formatted = raw.map(format_example, remove_columns=raw.column_names)

# quick leakage sanity check: flag rows where the assistant turn looks like full code
def looks_like_code_dump(example, idx):
    msgs = raw[idx]["messages"]
    a = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
    suspicious = ("```" in a) or ("def " in a and "return" in a) or a.count("\n") > 8
    return suspicious

suspicious_idx = [i for i in range(len(raw)) if looks_like_code_dump(None, i)]
print(f"Rows that look like they may contain full code (review these): {len(suspicious_idx)}")
if suspicious_idx[:5]:
    print("Example flagged indices:", suspicious_idx[:5])

split = formatted.train_test_split(test_size=cfg.val_split_ratio, seed=cfg.seed)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print("\n--- Sample formatted example ---\n")
print(train_dataset[0]["text"][:800])

## 4. Baseline evaluation (before fine-tuning)
Generate hint responses from the untouched quantized base model on a few held-out prompts, so we have a real before/after comparison.

In [ ]:
def build_eval_prompts(eval_ds, raw_ds, n=8):
    """Pull the system+user turns from n eval examples to use as generation prompts."""
    # Recover original message lists for eval examples via index alignment on 'text'
    text_to_msgs = {}
    for ex in raw_ds:
        t = tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
        text_to_msgs[t] = ex["messages"]
    prompts = []
    for row in eval_ds.select(range(min(n, len(eval_ds)))):
        msgs = text_to_msgs.get(row["text"])
        if msgs is None:
            continue
        convo = [m for m in msgs if m["role"] in ("system", "user")]
        prompts.append(convo)
    return prompts

eval_prompts = build_eval_prompts(eval_dataset, raw, n=8)
print(f"Built {len(eval_prompts)} evaluation prompts")

In [ ]:
@torch.no_grad()
def generate_response(model, convo_messages):
    prompt = tokenizer.apply_chat_template(convo_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=cfg.max_new_tokens,
        temperature=cfg.gen_temperature,
        top_p=cfg.gen_top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen_only = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_only, skip_special_tokens=True).strip()

baseline_outputs = []
model.eval()
for convo in eval_prompts:
    resp = generate_response(model, convo)
    baseline_outputs.append(resp)

for i, (convo, resp) in enumerate(zip(eval_prompts, baseline_outputs)):
    user_msg = next(m["content"] for m in convo if m["role"] == "user")
    print(f"[{i}] USER: {user_msg}\nBASE MODEL: {resp}\n{'-'*80}")

## 5. Attach LoRA adapters (QLoRA setup)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.gradient_checkpointing)

peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias=cfg.lora_bias,
    task_type="CAUSAL_LM",
    target_modules=cfg.lora_target_modules,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type=cfg.lr_scheduler_type,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.max_grad_norm,
    optim=cfg.optim,
    logging_steps=cfg.logging_steps,
    eval_strategy=cfg.eval_strategy,
    save_strategy=cfg.save_strategy,
    save_total_limit=cfg.save_total_limit,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    max_seq_length=cfg.max_seq_length,
    packing=cfg.packing,
    dataset_text_field="text",
    gradient_checkpointing=cfg.gradient_checkpointing,
    report_to="none",
    seed=cfg.seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

train_result = trainer.train()
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(train_result)

## 7. Evaluate: quantitative loss/perplexity

In [ ]:
import math

metrics = trainer.evaluate()
eval_loss = metrics["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Fine-tuned model — eval_loss: {eval_loss:.4f} | perplexity: {perplexity:.2f}")
metrics

## 8. Evaluate: qualitative comparison (base vs fine-tuned)
Re-run generation on the same held-out prompts with the fine-tuned (LoRA-attached) model, and check a simple heuristic for whether the response leaks full solution code (presence of code fences / multi-line function bodies) vs. gives a hint.

In [ ]:
model.eval()
model.config.use_cache = True

finetuned_outputs = []
for convo in eval_prompts:
    resp = generate_response(model, convo)
    finetuned_outputs.append(resp)

def leakage_flag(text: str) -> bool:
    """Heuristic: full code dump vs. a hint."""
    return ("```" in text) or ("def " in text and "return" in text and text.count("\n") > 4)

print(f"{'idx':<4}{'base leaked?':<15}{'finetuned leaked?':<18}")
for i, (b, f) in enumerate(zip(baseline_outputs, finetuned_outputs)):
    print(f"{i:<4}{str(leakage_flag(b)):<15}{str(leakage_flag(f)):<18}")

base_leak_rate = sum(leakage_flag(o) for o in baseline_outputs) / len(baseline_outputs)
ft_leak_rate = sum(leakage_flag(o) for o in finetuned_outputs) / len(finetuned_outputs)
print(f"\nBase model solution-leak rate:       {base_leak_rate:.0%}")
print(f"Fine-tuned model solution-leak rate: {ft_leak_rate:.0%}")

In [ ]:
for i, (convo, b, f) in enumerate(zip(eval_prompts, baseline_outputs, finetuned_outputs)):
    user_msg = next(m["content"] for m in convo if m["role"] == "user")
    print(f"[{i}] USER: {user_msg}")
    print(f"  BASE      : {b[:300]}")
    print(f"  FINE-TUNED: {f[:300]}")
    print("-" * 100)

## 9. Save / export the LoRA adapter
The adapter (a few MB) is saved to `cfg.output_dir`. To deploy, load the base model in 4-bit and attach this adapter with `PeftModel.from_pretrained`, or merge with `merge_and_unload()` for a standalone fp16 model.

In [ ]:
print(f"Adapter + tokenizer saved to: {cfg.output_dir}")
!ls -la {cfg.output_dir}

# Optional: zip for download from Colab
import shutil
shutil.make_archive(cfg.output_dir, 'zip', cfg.output_dir)
print(f"Zipped adapter available at: {cfg.output_dir}.zip")

## 10. (Reference) Loading the fine-tuned model later
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", quantization_config=bnb_config, device_map="auto")
tok = AutoTokenizer.from_pretrained("qwen2.5-1.5b-hint-qlora")
model = PeftModel.from_pretrained(base, "qwen2.5-1.5b-hint-qlora")
```